In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
DATA_DIR = "../data/b_cb/raw"
OUT_DIR = "../data/b_cb/processed"
os.makedirs(OUT_DIR, exist_ok=True)

DELTA_PHASE_B = 0.036
DELTA_PHASE_Cb = 0.041
DELTA_EPS = 1e-6
DOWNSAMPLE = 20

In [3]:
def parse_nso(line: str):
    """
    Parse a global-observable line starting with '# Nso'.

    Example input line:
        # Nso 16344 104277 253668 110490 100028 1.163670 3740

    According to the CDT data format, we:
      - take all numeric values after 'Nso'
      - drop the last two values
      - keep the very last value
    This selects the subset of global observables used in the paper.

    Returns:
        List[float]: selected global observables
    """
    # split line into tokens
    parts = line.split()

    # find the position of the 'Nso' keyword
    idx = parts.index("Nso")

    # take everything after 'Nso'
    after = parts[idx + 1:]

    # keep all but the last two entries, and also keep the final entry
    values = after[:-2] + after[-1:]

    # convert all values to float
    return [float(v) for v in values]


def parse_vto(line: str):
    """
    Parse a local-observable line starting with 'Vto'.

    Example input line:
        Vto t x1 x2 x3 x4 x5 x6

    The first two entries ('Vto' and time index t) are discarded.
    Only the six local geometric observables are returned.

    Returns:
        List[float]: local observables for a single time slice
    """
    # split line into tokens
    parts = line.split()

    # skip 'Vto' and time index, keep x1..x6
    return [float(v) for v in parts[2:]]


def parse_delta_from_filename(filename: str) -> float:
    return float(filename.split("-")[2])


def flatten_sample(sample):
    """
    Convert a single CDT configuration into a flat feature vector.

    The feature vector consists of:
      - global observables (Nso)
      - local observables (Vto), ordered by discrete time slice

    This ordering enforces time-translation symmetry after
    cyclic time-shift augmentation.

    Returns:
        List[float]: 1D feature vector (length = 30)
    """
    features = []

    # add global observables
    features.extend(sample["Nso"])

    # add local observables in time order
    for vto_t in sample["Vto"]:
        features.extend(vto_t)

    return features

# ntime 4 59999 termal
# Nso 15039 96076 233914 101918 99864 1.216083 15704
# v 39.524725 129.171952 1.015896 5.405741
# Vto 1 30429 288 296 3469 9149 10499
# Vto 2 3469 253 223 4121 1034 15416
# Vto 3 4121 277 263 11913 1249 10460
# Vto 4 11913 232 222 30429 3607 15704


# ntime 4 486097 fixed
# Nso 16337 103876 252134 109730 99950 1.172670 4246
# v 13.727677 44.949225 1.015695 23.922179
# K0 4.730000 Delta 0.600000 K4 1.172670
# cid 4 cnt 478781 xid 11
# Vto 1 12968 1146 1046 12053 4246 4103
# Vto 2 12053 1289 1235 12564 3940 3540
# Vto 3 12564 1217 1271 12390 4105 3138
# Vto 4 12390 1263 1313 12968 4046 4246

def label_from_delta(delta):
    """
    Assign a phase label based on κ₀.

    Deep inside phase C:
        label = 0
    Deep inside phase A:
        label = 1
    Intermediate κ₀ values:
        label = None (not used for training)

    A small tolerance is used to avoid floating-point issues.

    Returns:
        int or None
    """
    if abs(delta - DELTA_PHASE_B) < DELTA_EPS:
        return 0
    if abs(delta - DELTA_PHASE_Cb) < DELTA_EPS:
        return 1
    return None

In [4]:
files = sorted(
    f for f in os.listdir(DATA_DIR)
    if f.startswith("vto-") and f.endswith(".out")
)

In [5]:
files

['vto-2.2-0.036-100k-T4-torus.out-L.out',
 'vto-2.2-0.037-100k-T4-torus.out-L.out',
 'vto-2.2-0.038-100k-T4-torus.out-L.out',
 'vto-2.2-0.039-100k-T4-torus.out-L.out',
 'vto-2.2-0.040-100k-T4-torus.out-L.out',
 'vto-2.2-0.041-100k-T4-torus.out-L.out']

In [6]:
# Containers for the final dataset (filled after concatenation)
X_full = []
y_full = []
Delta_full = []

# Separate buffers for each time-shift variant
# shift = 0, 1, 2, 3 correspond to cyclic time translations
X_shifts = [[], [], [], []]
y_shifts = [[], [], [], []]
Delta_shifts = [[], [], [], []]


# Loop over all CDT output files (each file corresponds to a fixed delta)
for file_idx, filename in enumerate(tqdm(files, desc="Parsing files")):

    # Number of initial Monte Carlo configurations to discard
    # (thermalization cut; endpoints use a slightly smaller cut)
    SKIP_SAMPLES = 100_000 if file_idx in (0, len(files) - 2, len(files) - 1) else 200_000

    # Extract delta value from filename and assign phase label (if deep A or B)
    file_delta = parse_delta_from_filename(filename)
    file_label = label_from_delta(file_delta)

    file_path = os.path.join(DATA_DIR, filename)

    # Temporary storage for the currently parsed configuration
    current_sample = None
    sample_counter = 0

    # Read the file line by line
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            # Marker for the beginning of a new configuration
            if line.startswith("# ntime"):

                # If a previous configuration was fully read, process it
                if current_sample is not None:
                    sample_counter += 1

                    # Skip non-equilibrated configurations
                    if sample_counter > SKIP_SAMPLES:

                        # Apply cyclic time-shift augmentation (×4)
                        for shift in range(4):
                            perm = {
                                # Global observables are unchanged by time shifts
                                "Nso": current_sample["Nso"],

                                # Local observables are cyclically shifted in time
                                "Vto": (
                                    current_sample["Vto"][shift:]
                                    + current_sample["Vto"][:shift]
                                )
                            }

                            # Store features, κ₀ value, and label
                            X_shifts[shift].append(flatten_sample(perm))
                            Delta_shifts[shift].append(file_delta)
                            y_shifts[shift].append(file_label)

                # Initialize a new configuration container
                current_sample = {
                    "Nso": None,   # global observables
                    "Vto": []      # list of local observables (one per time slice)
                }

            # Parse global observables
            elif line.startswith("# Nso"):
                current_sample["Nso"] = parse_nso(line)

            # Parse local observables for a single time slice
            elif line.startswith("Vto"):
                current_sample["Vto"].append(parse_vto(line))

    # Report number of equilibrated configurations processed in this file
    print(f"{filename}: parsed {sample_counter - SKIP_SAMPLES} samples")


# After all files are processed, concatenate time-shift buffers
# The final ordering is:
#   all shift-0 samples, then all shift-1, shift-2, shift-3 samples
X_full = np.concatenate(
    [np.asarray(X_shifts[s]) for s in range(4)],
    axis=0
)

Delta_full = np.concatenate(
    [np.asarray(Delta_shifts[s]) for s in range(4)],
    axis=0
)

y_full = np.concatenate(
    [np.asarray(y_shifts[s], dtype=object) for s in range(4)],
    axis=0
)


Parsing files:  17%|█████                         | 1/6 [00:01<00:06,  1.25s/it]

vto-2.2-0.036-100k-T4-torus.out-L.out: parsed 12738 samples


Parsing files:  33%|██████████                    | 2/6 [00:04<00:08,  2.15s/it]

vto-2.2-0.037-100k-T4-torus.out-L.out: parsed 35911 samples


Parsing files:  50%|███████████████               | 3/6 [00:06<00:07,  2.40s/it]

vto-2.2-0.038-100k-T4-torus.out-L.out: parsed 36189 samples


Parsing files:  83%|█████████████████████████     | 5/6 [00:10<00:02,  2.13s/it]

vto-2.2-0.039-100k-T4-torus.out-L.out: parsed 17431 samples
vto-2.2-0.040-100k-T4-torus.out-L.out: parsed 20233 samples


Parsing files: 100%|██████████████████████████████| 6/6 [00:12<00:00,  2.07s/it]

vto-2.2-0.041-100k-T4-torus.out-L.out: parsed 28039 samples


In [7]:
X_full = np.asarray(X_full)
y_full = np.asarray(y_full, dtype=object)
Delta_full = np.asarray(Delta_full)

np.savez(
    os.path.join(OUT_DIR, "dataset_full_bcb.npz"),
    X=X_full,
    y=y_full,
    Delta=Delta_full
)

print("dataset_full_bcb.npz zapisany")
print("X_full shape:", X_full.shape)


dataset_full_bcb.npz zapisany
X_full shape: (602164, 30)


In [8]:
mask_train = (
    (Delta_full == DELTA_PHASE_B) |
    (Delta_full == DELTA_PHASE_Cb)
) & (y_full != None)

X_train = X_full[mask_train]
y_train = y_full[mask_train].astype(int)
Delta_train = Delta_full[mask_train]

# downsampling
idx = np.arange(len(X_train)) % DOWNSAMPLE == 0

X_train = X_train[idx]
y_train = y_train[idx]
Delta_train = Delta_train[idx]

np.savez(
    os.path.join(OUT_DIR, "dataset_train_shortened_bcb.npz"),
    X=X_train,
    y=y_train,
    Delta=Delta_train
)

print("dataset_train_shortened_bcb.npz zapisany")
print("X_train shape:", X_train.shape)


dataset_train_shortened_bcb.npz zapisany
X_train shape: (8156, 30)
